In [2]:
import torch
import numpy as np

In [3]:
class MSELoss:
    def forward(pred: np.ndarray, target: np.ndarray) -> float:

        self.E = target - pred
        R = self.E**2
        return np.sum(R) / R.size

    def bacward(self) -> np.ndarray:
        return -2 * self.E / self.E.size

    def __call__(self, pred: np.ndarray, target: np.ndarray) -> float:
        return self.forward(pred, target)

In [4]:
class CrossEntropyLoss:

    def forward(self, pred: np.ndarray, target: np.ndarray) -> float:
        self.batch_size = pred.shape[0]
        self.target = target

        logits_max = pred.max(axis=1, keepdims=True)
        exp_pred = np.exp(pred - logits_max)

        self.softmax = exp_pred / exp_pred.sum(axis=1, keepdims=True)

        log_sum_exp = np.log(exp_pred.sum(axis=1, keepdims=True))
        loss = (-np.sum(target * pred, axis=1, keepdims=True) + logits_max + log_sum_exp)

        return loss.mean()

    def backward(self) -> np.ndarray:
        return (self.softmax - self.target) / self.batch_size

    def __call__(self, pred: np.ndarray, target: np.ndarray) -> float:
        return self.forward(pred, target)

In [5]:
class BCEWithLogitsLoss:
    def forward(self, pred: np.ndarray, target: np.ndarray) -> float:
        self.N = pred.size
        self.target = target

        self.sigm = 1 / (1 + np.exp(-pred))
        r = self.relu(-pred)
        bceloss = (
            (1 - target) * pred + r + np.log(np.exp(-r) + np.exp(-(pred + r)))
        )

        return bceloss.mean()

    def backward(self) -> np.ndarray:
        return (self.sigm - self.target)/ self.N

    def relu(sefl, pred: np.ndarray) -> np.ndarray:
        return np.maximum(0, pred)

    def __call__(self, pred: np.ndarray, target: np.ndarray) -> float:
        return self.forward(pred, target)

In [12]:
pred = torch.randn(16, 10, dtype=torch.float32, requires_grad=True)
target = torch.randn(16, 10).softmax(dim=1)

loss_func = torch.nn.CrossEntropyLoss()
l = loss_func(pred, target)
l.backward()

m_loss_func = CrossEntropyLoss()
m_loss = m_loss_func(pred.detach().numpy(), target.numpy())

my_grad = m_loss_func.backward()

print(f"torch: {l.item()}")
print(f"my_torch: {m_loss.item()}")
print(target.shape)
print(f"torch_grad: {np.allclose(pred.grad.detach().numpy(), my_grad)}")

torch: 2.683913230895996
my_torch: 2.683913469314575
torch.Size([16, 10])
torch_grad: True


In [11]:
pred = torch.randn(16, 10, dtype=torch.float32, requires_grad=True)
target = torch.randn(16, 10).softmax(dim=1)

loss_func = torch.nn.BCEWithLogitsLoss()
l = loss_func(pred, target)
l.backward()

m_loss_func = BCEWithLogitsLoss()
m_loss = m_loss_func(pred.detach().numpy(), target.numpy())

my_grad = m_loss_func.backward()

print(f"torch: {l.item()}")
print(f"my_torch: {m_loss.item()}")

print(f"torch_grad: {np.allclose(pred.grad.detach().numpy(), my_grad)}")

torch: 0.7729238867759705
my_torch: 0.7729238867759705
torch_grad: True
